# Как работать с предобученными CNN в PyTorch

Этот ноутбук даёт пошаговое понимание:
- откуда брать предобученные модели;
- что такое «голова» сети и зачем её снимать;
- какой размер входа и какой размер эмбеддинга на выходе;
- как правильно препроцессить картинку под конкретную модель.

## Ликбез: задачи компьютерного зрения и предобученные модели в PyTorch

Ниже — краткий обзор основных задач CV и того, что есть в **torchvision** (и рядом с ним) с готовыми весами.

### Основные задачи CV

| Задача | Что делаем | Пример |
|--------|------------|--------|
| **Классификация изображений** | По картинке предсказываем один класс (кошка, собака, болезнь и т.д.) | ResNet, EfficientNet, VGG |
| **Детекция объектов (Object Detection)** | Находим на изображении объекты и рисуем вокруг них боксы (bbox), у каждого — класс | Faster R-CNN, RetinaNet, FCOS |
| **Сегментация (Semantic)** | Для каждого пикселя предсказываем класс (дорога, небо, человек) | FCN, DeepLabV3 |
| **Сегментация (Instance)** | Отдельно выделяем каждый объект: и бокс, и маска пикселей | Mask R-CNN |
| **Оценка позы (Pose / Keypoints)** | Находим людей и ключевые точки (суставы, голова) | Keypoint R-CNN |
| **Поиск похожих изображений** | По картинке ищем похожие в базе (через эмбеддинги) | Любая классификационная CNN без головы |

### Что есть в torchvision и на чём обучено

| Задача | Модуль / пример модели | Датасет обучения | Кратко про датасет |
|--------|------------------------|------------------|---------------------|
| **Классификация** | `models.resnet18`, `efficientnet_b0`, `mobilenet_v2`, `vgg16` | **ImageNet (ILSVRC 2012)** | ~1,2 млн фото, 1000 классов (объекты, животные, сцены) |
| **Детекция** | `models.detection.fasterrcnn_resnet50_fpn`, `retinanet_resnet50_fpn`, `fcos_resnet50_fpn` | **COCO** | Обычно COCO (80 классов: человек, машина, собака, стул и т.д.), боксы и классы |
| **Instance segmentation** | `models.detection.maskrcnn_resnet50_fpn` | **COCO** | Те же 80 классов + маска по каждому объекту |
| **Semantic segmentation** | `models.segmentation.fcn_resnet50`, `deeplabv3_resnet50` | **PASCAL VOC** или **COCO** | Класс на каждый пиксель (фон, человек, машина, небо и т.д.) |
| **Keypoints (поза)** | `models.detection.keypointrcnn_resnet50_fpn` | **COCO** | Детекция людей + 17 ключевых точек (голова, плечи, колени и т.д.) |

На олимпиадах по ИИ чаще всего используют **классификационные** модели как фичевыделители (эмбеддинги + свой классификатор); детекция и сегментация обычно не обязательны. Дальше в ноутбуке разбираем как раз классификацию и эмбеддинги.

In [ ]:
# В torchvision модели разнесены по подмодулям:
# models.* — классификация (resnet18, efficientnet_b0, ...)
# models.detection — детекция и instance segmentation (fasterrcnn_*, maskrcnn_*, keypointrcnn_*)
# models.segmentation — семантическая сегментация (fcn_*, deeplabv3_*)

from torchvision import models

print("Классификация (примеры):", [x for x in dir(models) if not x.startswith("_") and "resnet" in x.lower()][:4])
print("Детекция:", [x for x in dir(models.detection) if not x.startswith("_")][:6])
print("Сегментация:", [x for x in dir(models.segmentation) if not x.startswith("_")][:6])

### Примеры изображений из датасетов

Ниже — по одному примеру картинки из каждого датасета (загружаются по URL при выполнении ячейки), чтобы было наглядно, как выглядят данные. Для PASCAL VOC показана типичная сцена с объектами в контексте; при наличии интернета ячейка подгрузит изображения.

In [ ]:
# Примеры картинок из датасетов (загрузка по URL)
from urllib.request import urlopen
from IPython.display import display, Image as IPImage

def show_from_url(url, title="", width=300):
    try:
        data = urlopen(url, timeout=10).read()
        display(IPImage(data=data, width=width))
    except Exception as e:
        print(f"{title}: не удалось загрузить ({e})")

# ImageNet: типичный объект из 1000 классов (кошка)
print("ImageNet (ILSVRC 2012) — один из 1000 классов:")
show_from_url(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/480px-Cat03.jpg",
    "ImageNet",
    width=320
)

# COCO: сцена с несколькими объектами (люди, стулья, телевизор и т.д.)
print("\nCOCO — сцена с объектами (80 классов):")
show_from_url(
    "https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000139.jpg",
    "COCO",
    width=320
)

# PASCAL VOC: сцена с объектами и сегментацией (20 классов)
print("\nPASCAL VOC — сцена с объектами для детекции/сегментации:")
show_from_url(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2d/Bicycle_race_-_Mountain_bike_-_Marathon_du_Mont-Blanc_2015.jpg/640px-Bicycle_race_-_Mountain_bike_-_Marathon_du_Mont-Blanc_2015.jpg",
    "PASCAL VOC (похожий тип сцен: объекты в контексте)",
    width=320
)

## 1. Откуда брать предобученные модели

В PyTorch предобученные модели для картинок живут в **`torchvision.models`**. При первом вызове веса скачиваются с сервера PyTorch и кэшируются локально.

Примеры: `resnet18`, `resnet50`, `efficientnet_b0`, `mobilenet_v2`, `vgg16` и др. У каждой — свои веса (например, обучение на ImageNet).

In [ ]:
import torch
from torchvision import models

# Загружаем ResNet18 с весами, обученными на ImageNet
model_full = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print(type(model_full))
print(model_full)

В конце вы видите слой **`(fc): Linear(in_features=512, out_features=1000)`**. Это и есть **«голова» (head)**: полносвязный слой, который выдаёт 1000 чисел — вероятности классов ImageNet. Для своих задач нам нужны не эти 1000 классов, а **вектор признаков (эмбеддинг)** перед головой — размерности 512.

## 2. Как устроена модель: где «тело» и где «голова»

Удобно посмотреть список дочерних модулей. У ResNet последний элемент — как раз `fc`.

In [ ]:
# Список «блоков» модели
children = list(model_full.children())
print("Number of top-level modules:", len(children))
for i, m in enumerate(children):
    print(i, m)

**Снятие головы:** берём все слои **кроме последнего** — так мы оставляем только «тело» сети, которое выдаёт вектор признаков фиксированной длины.

In [ ]:
# Модель без последнего слоя (без fc)
feature_extractor = torch.nn.Sequential(*list(model_full.children())[:-1])

print("Feature extractor (без головы):")
print(feature_extractor)

## 3. Какой размер входа и какой размер выхода (эмбеддинга)

- **Вход:** почти все модели из `torchvision` обучены на ImageNet и ожидают картинки **224×224** пикселей (иногда 299 для Inception). Формат тензора: `(batch, 3, 224, 224)`.
- **Нормализация:** те же mean и std, что и при обучении на ImageNet (см. ниже).
- **Выход** после снятия головы у ResNet18: тензор формы `(batch, 512, 1, 1)`. Чтобы получить вектор размера 512, делаем `.flatten()` или `.view(batch, -1)`.

In [ ]:
from torchvision import transforms
from PIL import Image

# Размер входа для ResNet (и большинства моделей ImageNet)
IMG_SIZE = 224

# Нормализация ImageNet — обязательна, иначе качество падает
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Проверим на одной картинке (случайный тензор для примера)
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)  # batch=2
feature_extractor.eval()
with torch.no_grad():
    out = feature_extractor(x)

print("Вход:  x.shape =", x.shape)
print("Выход (до flatten): out.shape =", out.shape)
out_flat = out.view(out.size(0), -1)
print("Выход (после view): out_flat.shape =", out_flat.shape)
print("Размерность эмбеддинга на одну картинку:", out_flat.shape[1])

## 4. Сводка по популярным моделям (torchvision)

| Модель           | Вход (H, W) | Размер эмбеддинга | Лучше всего для задач CV | На чём обучены (примеры датасетов) |
|------------------|-------------|-------------------|---------------------------|-------------------------------------|
| ResNet18         | 224×224     | 512               | Классификация изображений, поиск похожих картинок, эмбеддинги под свой классификатор; быстрый старт и прототипы | **ImageNet (ILSVRC 2012)** — ~1,2 млн фото, 1000 классов (животные, объекты, сцены) |
| ResNet50         | 224×224     | 2048              | То же, что ResNet18, когда нужна выше точность; фичевыделитель для детекции/сегментации | **ImageNet (ILSVRC 2012)** |
| EfficientNet_B0  | 224×224     | 1280              | Классификация и эмбеддинги при балансе точность/скорость; современный выбор для соревнований | **ImageNet (ILSVRC 2012)** |
| MobileNet_V2     | 224×224     | 1280              | Классификация и поиск на телефонах и edge-устройствах; малый размер модели и быстрый inference | **ImageNet (ILSVRC 2012)** |
| VGG16            | 224×224     | 512×7×7 → 25088   | Классификация, эмбеддинги; простая архитектура, но тяжёлая и медленная; после снятия головы нужен flatten | **ImageNet (ILSVRC 2012)** |

**Про датасет ImageNet:** это большой набор изображений (ILSVRC 2012): объекты, животные, сцены — 1000 классов. Предобучение на нём даёт модели общие визуальные признаки, которые хорошо переносятся на другие задачи (другие классы, домены).

У VGG после снятия последнего `fc` остаётся объём 512×7×7 — его нужно вытянуть в вектор (например, 25088). У ResNet и многих других в конце уже есть global average pooling, поэтому на выходе (C, 1, 1) — сразу вектор длины C.

## 5. Альтернативный способ снять голову: заменить fc на Identity

Вместо того чтобы собирать новый `Sequential` из `children()[:-1]`, можно оставить модель как есть и **заменить последний слой на пустой** (ничего не делает). Тогда архитектура одна и та же, но на выходе — те же 512 чисел (для ResNet18).

In [ ]:
model_full2 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# У ResNet атрибут fc — последний полносвязный слой
print("До замены: model_full2.fc =", model_full2.fc)

# Заменяем на слой «как есть»: выход не меняется
model_full2.fc = torch.nn.Identity()
print("После замены: model_full2.fc =", model_full2.fc)

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = model_full2(x)
print("Выход после Identity: shape =", out.shape)

Удобно: не нужно вручную собирать `Sequential` и помнить, какой индекс последнего слоя. Но имя атрибута (`fc`, `classifier` и т.д.) у каждой архитектуры своё — это можно посмотреть в документации или вывести `model`.

## 6. ResNet18 vs ResNet50: сравнение размерности

Проверим на одном и том же входе.

In [ ]:
r18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
r18 = torch.nn.Sequential(*list(r18.children())[:-1])

r50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
r50 = torch.nn.Sequential(*list(r50.children())[:-1])

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    e18 = r18(x).view(1, -1)
    e50 = r50(x).view(1, -1)

print("ResNet18 embedding dim:", e18.shape[1])
print("ResNet50 embedding dim:", e50.shape[1])

## 7. Полный пример: одна картинка → вектор признаков

Загружаем изображение с диска (или подставляем путь к своей картинке), препроцессим и получаем эмбеддинг.

In [ ]:
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model = torch.nn.Sequential(*list(model.children())[:-1])
model = model.eval().to(device)

def image_to_embedding(image_path, model, transform, device):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
    return out.cpu().view(1, -1).numpy()

# Пример: если есть папка data_pets/train, берём первую картинку
data_dir = Path("data_pets") / "train"
if data_dir.exists():
    first_image = next(data_dir.glob("*.jpg"), None) or next(data_dir.glob("*.png"), None)
    if first_image is not None:
        emb = image_to_embedding(first_image, model, transform, device)
        print("Path:", first_image)
        print("Embedding shape:", emb.shape)
        print("First 5 values:", emb[0, :5])
    else:
        print("Нет картинок в data_pets/train — подставьте свой путь в first_image")
else:
    print("Папка data_pets/train не найдена. Подставьте свой путь к картинке.")
    # emb = image_to_embedding("path/to/your/image.jpg", model, transform, device)

## 8. Пример на данных соревнования Pet Emotions (Kaggle)

Ниже мы применяем всё, что разобрали выше, к реальным данным нашего соревнования: папки `data_pets/train` и `data_pets/test`, таблицы `train.csv` и `test.csv` с метками классов эмоций питомцев (Angry, Sad, happy, Other).

Шаги:
1. Загружаем фичевыделитель (ResNet18 без головы) и препроцессинг.
2. Считаем эмбеддинги для всех обучающих картинок.
3. Обучаем логистическую регрессию на эмбеддингах.
4. Считаем эмбеддинги для тестовых картинок и получаем предсказания.
5. Смотрим accuracy на train и пример предсказаний на test.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Путь к данным соревнования: папка data_pets должна лежать в текущей рабочей директории
# (запускайте ноутбук из папки, где лежит data_pets, или укажите полный путь)
DATA = Path("data_pets")
assert DATA.exists(), "Папка data_pets не найдена. Запустите ячейки из раздела 3 (transform) и поместите data_pets рядом с ноутбуком."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Фичевыделитель и препроцессинг — как в разделах выше
extractor = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
extractor = torch.nn.Sequential(*list(extractor.children())[:-1])
extractor = extractor.eval().to(device)

def get_embeddings(model, paths):
    model.eval()
    embs = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
            embs.append(model(x).cpu().view(1, -1).numpy())
    return np.vstack(embs)

# Train
train_df = pd.read_csv(DATA / "train.csv")
train_paths = [DATA / "train" / f for f in train_df["image"]]
y_train = train_df["label"].values

print("Считаем эмбеддинги для train...")
X_train = get_embeddings(extractor, train_paths)
print("X_train.shape:", X_train.shape)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
print("Train accuracy:", clf.score(X_train, y_train))

# Test (можно взять только первые 50 для скорости, или все)
test_df = pd.read_csv(DATA / "test.csv")
test_paths = [DATA / "test" / f for f in test_df["image"]]
print("Считаем эмбеддинги для test...")
X_test = get_embeddings(extractor, test_paths)
preds = clf.predict(X_test)

print("\nПример предсказаний на test:")
print(pd.DataFrame({"image": test_df["image"][:10], "predicted": preds[:10]}))

## 9. Что запомнить

- **Где брать:** `torchvision.models` (resnet18, resnet50, efficientnet_b0, mobilenet_v2, …).
- **Снять голову:** либо `Sequential(*list(model.children())[:-1])`, либо заменить `model.fc` на `Identity()` (для ResNet).
- **Вход:** обычно 224×224, нормализация ImageNet (mean/std из документации).
- **Выход:** после снятия головы — тензор (batch, C, 1, 1) или (batch, C, H, W); для классификатора нужен вектор — делаем `.view(batch, -1)`.
- **Размерность C** у каждой модели своя (ResNet18 → 512, ResNet50 → 2048 и т.д.) — смотри таблицу или проверяй в коде.